<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/03_Advanced/L31_Training_From_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L31: Training From Scratch - Building a Small LM

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Advanced  
**Lesson:** 31 of 45

---

## Learning Objectives

By the end of this lesson, you will:
- Prepare raw text data for language model training
- Initialize and train a small causal LM from scratch (or minimal config)
- Use checkpointing and basic training monitoring
- Understand data preparation, training loops, and model initialization

---

## Concept: Training From Scratch

Training from scratch requires: (1) **Data**: tokenized corpus, possibly chunked to fixed length; (2) **Model**: small transformer or GPT-2 config; (3) **Training loop**: forward, loss (cross-entropy), backward, optimizer step; (4) **Checkpointing**: save model/optimizer periodically.

---

In [ ]:
!pip install transformers torch datasets -q
from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM, Trainer, TrainingArguments
from datasets import load_dataset
import torch
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded.")

## Part 1: Data Preparation

Load text, tokenize, and create fixed-length chunks for causal LM (predict next token).

---

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

def tokenize_and_chunk(examples, block_size=128):
    concatenated = tokenizer(examples["text"], return_tensors="pt", truncation=True, max_length=block_size)
    return {"input_ids": concatenated["input_ids"].squeeze(0), "attention_mask": concatenated["attention_mask"].squeeze(0)}

dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train", trust_remote_code=True)
dataset = dataset.filter(lambda x: len(x["text"].strip()) > 0)
dataset = dataset.map(lambda x: tokenize_and_chunk({"text": x["text"]}), remove_columns=dataset.column_names, num_proc=1)
dataset.set_format("torch")
print(f"Samples: {len(dataset)}")

## Part 2: Model Initialization (Small Config)

Use a small GPT-2 config or create from config to train from scratch.

---

In [ ]:
from transformers import GPT2Config, GPT2LMHeadModel

config = GPT2Config(
    vocab_size=tokenizer.vocab_size,
    n_positions=128,
    n_embd=256,
    n_layer=4,
    n_head=4,
)
model = GPT2LMHeadModel(config)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## Part 3: Training Loop with Checkpointing

---

In [ ]:
args = TrainingArguments(
    output_dir="./out_scratch_l31",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    save_steps=100,
    save_total_limit=2,
    logging_steps=50,
    report_to="none",
)
trainer = Trainer(model=model, args=args, train_dataset=dataset)
trainer.train()
trainer.save_model("./out_scratch_l31/final")
print("Training done. Checkpoints saved.")

## Exercises

1. Increase block_size and n_layer; compare convergence and memory.
2. Add validation split and log eval loss.
3. Implement a simple custom training loop with manual checkpointing every N steps.

---

## Key Takeaways

1. Data: tokenize and chunk to fixed length; labels = input_ids for causal LM.
2. Model: initialize from config for true from-scratch; or start from pre-trained and continue.
3. Checkpointing and logging are essential for long runs.

---

## Next Lesson

**L32: Custom Architectures** – Custom attention and transformer variants.

---